# 02. Construcción de la base de datos de árboles de ingredientes

**Aplicación en el proyecto:** a partir de `UpdatedProductIngredients_11_15.json` y `GroceryDB_foods.csv` se construye la base de datos con la que se entrenarán las GNN, cumpliendo tres reglas:

1. **Cruzar por `original_ID`** con el CSV únicamente para rescatar la etiqueta `f_FPro`. Las columnas de nutrientes del CSV no se usan, porque el FPro se calculó a partir de ellas (evita la fuga de datos).
2. **Aplanar el `ingredient_tree`** para que cada ingrediente y subingrediente quede registrado con su padre (`id_padre`), su profundidad y su posición (`order`).
3. **Conservar los campos del JSON** (`additive_id`, `ingredient_type`) para decidir más adelante si se usan como atributos de los nodos.

Cada nodo tiene un `id_nodo` numérico dentro de su producto (0 para la raíz). Así, si un ingrediente como `water` aparece dos veces en una receta, son dos nodos distintos.

## 1. Carga del CSV y del JSON

In [1]:
import json
import pandas as pd
from pathlib import Path

ruta_csv = "../data/GroceryDB_foods.csv"
ruta_json = "../data/UpdatedProductIngredients_11_15.json"

# 1. Columnas reales del CSV
df_completo = pd.read_csv(ruta_csv)
print("=== COLUMNAS DISPONIBLES EN EL CSV ===")
print(list(df_completo.columns))
print(f"\nTotal de filas en el CSV: {len(df_completo)}")

# 2. Columnas relacionadas con FPro y cuántos nulos tienen
columnas_fpro = [c for c in df_completo.columns if "fpro" in c.lower()]
print(f"Columnas relacionadas con FPro: {columnas_fpro}")
for col in columnas_fpro:
    print(f"  {col}: {df_completo[col].isna().sum()} valores nulos")

# 3. Cómo luce original_ID en el CSV
print("\nPrimeros 3 original_ID del CSV:")
print(df_completo["original_ID"].head(3).values)

=== COLUMNAS DISPONIBLES EN EL CSV ===
['original_ID', 'name', 'store', 'harmonized single category', 'brand', 'f_FPro', 'f_FPro_P', 'f_min_FPro', 'f_std_FPro', 'f_FPro_class', 'price', 'price percal', 'package_weight', 'has10_nuts', 'is_Nuts_Converted_100g', 'Protein', 'Total Fat', 'Carbohydrate', 'Sugars, total', 'Fiber, total dietary', 'Calcium', 'Iron', 'Sodium', 'Vitamin C', 'Cholesterol', 'Fatty acids, total saturated', 'Total Vitamin A']

Total de filas en el CSV: 50468
Columnas relacionadas con FPro: ['f_FPro', 'f_FPro_P', 'f_min_FPro', 'f_std_FPro', 'f_FPro_class']
  f_FPro: 23811 valores nulos
  f_FPro_P: 23811 valores nulos
  f_min_FPro: 24081 valores nulos
  f_std_FPro: 24081 valores nulos
  f_FPro_class: 23811 valores nulos

Primeros 3 original_ID del CSV:
['tg_15121683' 'tg_15121684' 'tg_53318168']


In [2]:
# De todo el CSV solo tomamos el ID y la etiqueta f_FPro.
# Los 12 nutrientes NO se usan: el FPro se calculó a partir de ellos.
df_csv = df_completo[["original_ID", "f_FPro"]].dropna()
print("IDs repetidos en el CSV (con FPro):", df_csv["original_ID"].duplicated().sum())

diccionario_fpro = dict(zip(df_csv["original_ID"], df_csv["f_FPro"]))
print(f"Productos con f_FPro válido en el CSV: {len(diccionario_fpro)}")

with open(ruta_json, "r", encoding="utf-8") as archivo:
    base_datos_grocery = json.load(archivo)
print(f"Productos totales en el JSON: {len(base_datos_grocery)}")

IDs repetidos en el CSV (con FPro): 0
Productos con f_FPro válido en el CSV: 26657
Productos totales en el JSON: 42541


## 2. Revisión de los datos antes de extraer

Antes de construir se comprueba cuántos productos tienen a la vez árbol de ingredientes y puntaje FPro (ese es el tamaño real del dataset), cómo se reparte el FPro entre las clases del CSV (para justificar el umbral de AUP) y cuántos productos tienen subingredientes anidados (para probar la jerarquía).

In [3]:
ids_json = {p["original_ID"] for p in base_datos_grocery}
print("Productos en el JSON:", len(base_datos_grocery), "| IDs únicos:", len(ids_json))
print("Con árbol de ingredientes vacío:",
      sum(1 for p in base_datos_grocery if not p.get("ingredient_tree")))
ids_con_arbol = {p["original_ID"] for p in base_datos_grocery if p.get("ingredient_tree")}
print("Con árbol y con FPro (los que usaremos):", len(ids_con_arbol & set(diccionario_fpro)))

# Rangos de FPro por clase, para justificar el umbral de AUP
df_cls = df_completo[["f_FPro", "f_FPro_class"]].dropna()
print("\nFPro por clase (f_FPro_class):")
print(df_cls.groupby("f_FPro_class")["f_FPro"].agg(["min", "max", "count"]))

# Productos con subingredientes anidados, para probar la jerarquía
con_sub = [p for p in base_datos_grocery
           if any(i.get("sub_ingredients") for i in p.get("ingredient_tree", []))]
print("\nProductos con subingredientes anidados:", len(con_sub))

Productos en el JSON: 42541 | IDs únicos: 42541
Con árbol de ingredientes vacío: 0
Con árbol y con FPro (los que usaremos): 22517

FPro por clase (f_FPro_class):
                   min       max  count
f_FPro_class                           
0.0           0.000000  0.499843   3327
1.0           0.326500  0.729093    620
2.0           0.294167  0.722582   2457
3.0           0.500005  0.999056  20253

Productos con subingredientes anidados: 18672


## 3. Función de extracción

Recorre recursivamente el `ingredient_tree` y devuelve una fila por nodo. La raíz (`id_nodo = 0`) no lleva el nombre del producto, para que ninguna información externa a los ingredientes llegue al modelo; el nombre queda solo en la tabla de productos. El `id_nodo` sigue el orden de lectura de la etiqueta, y `order` es la posición dentro de cada lista (se reinicia en cada nivel).

In [4]:
def extraer_datos_producto(producto_json, valor_fpro):
    """
    Aplana el 'ingredient_tree' de un producto en una lista de nodos con su
    conexión padre -> hijo, su profundidad y su 'order' dentro de la lista.
    Devuelve (resumen_del_producto, lista_de_nodos).
    """
    orig_id = producto_json.get("original_ID")
    nodos = []

    # Nodo 0: raíz del producto. No lleva el nombre del producto, para que
    # ninguna información externa a los ingredientes llegue al modelo.
    nodos.append({
        "original_ID": orig_id, "id_nodo": 0, "id_padre": None,
        "ingredient_name": "RAIZ_PRODUCTO", "general_name": "RAIZ_PRODUCTO",
        "ingredient_type": "root", "additive_id": None,
        "order": 0, "profundidad": 0,
    })

    contador = 1  # id_nodo sigue el orden de lectura de la etiqueta

    def recorrer(lista_ing, id_padre, nivel):
        nonlocal contador
        for ing in lista_ing:
            id_actual = contador
            contador += 1
            nodos.append({
                "original_ID": orig_id, "id_nodo": id_actual, "id_padre": id_padre,
                "ingredient_name": ing.get("ingredient_name"),
                "general_name": ing.get("general_name"),
                "ingredient_type": ing.get("ingredient_type"),
                "additive_id": ing.get("additive_id"),
                "order": ing.get("order"), "profundidad": nivel,
            })
            sub = ing.get("sub_ingredients", [])
            if sub:
                recorrer(sub, id_padre=id_actual, nivel=nivel + 1)

    recorrer(producto_json.get("ingredient_tree", []), id_padre=0, nivel=1)

    resumen = {
        "original_ID": orig_id,
        "product_name": producto_json.get("product_name", "Sin nombre"),
        "store": producto_json.get("store", "Desconocida"),
        "f_FPro": valor_fpro,
        "total_nodos": len(nodos),
        "profundidad_max": max(n["profundidad"] for n in nodos),
    }
    return resumen, nodos

## 4. Prueba del extractor

Se prueba con un árbol pequeño del que se conoce la respuesta, incluyendo subingredientes anidados. Si alguna comprobación falla, el extractor tiene un error.

In [5]:
# Árbol de juguete cuya respuesta conocemos
prueba = {"original_ID": "x", "product_name": "demo", "ingredient_tree": [
    {"ingredient_name": "A", "order": 1, "sub_ingredients": [
        {"ingredient_name": "A1", "order": 1, "sub_ingredients": []},
        {"ingredient_name": "A2", "order": 2, "sub_ingredients": []}]},
    {"ingredient_name": "B", "order": 2, "sub_ingredients": []}]}

info, nodos = extraer_datos_producto(prueba, 0.5)
assert info["total_nodos"] == 5 and info["profundidad_max"] == 2
assert [n["id_padre"] for n in nodos] == [None, 0, 1, 1, 0]
assert [n["profundidad"] for n in nodos] == [0, 1, 2, 2, 1]
print("Extractor OK")

Extractor OK


## 5. Corrida completa

Se procesan todos los productos que tienen árbol de ingredientes y puntaje FPro válido.

In [6]:
productos, nodos = [], []
for prod in base_datos_grocery:
    id_prod = prod.get("original_ID")
    if prod.get("ingredient_tree") and id_prod in diccionario_fpro:
        resumen, nodos_prod = extraer_datos_producto(prod, diccionario_fpro[id_prod])
        productos.append(resumen)
        nodos.extend(nodos_prod)

df_productos = pd.DataFrame(productos)
df_nodos = pd.DataFrame(nodos)
print("Productos en el dataset final:", len(df_productos))
print("Nodos totales:", len(df_nodos))

Productos en el dataset final: 22517
Nodos totales: 406330


## 6. Validaciones

Comprueban que la estructura es un árbol bien formado y que no hay columnas de nutrientes en los datos que recibirá el modelo. Si alguna falla, el notebook se detiene.

In [7]:
# Cada nodo es único dentro de su producto
assert not df_nodos.duplicated(["original_ID", "id_nodo"]).any()
# Cada producto tiene exactamente una raíz
assert (df_nodos["id_padre"].isna().groupby(df_nodos["original_ID"]).sum() == 1).all()
# El padre siempre aparece antes que el hijo (sin ciclos)
hijos = df_nodos[df_nodos["id_padre"].notna()]
assert (hijos["id_padre"] < hijos["id_nodo"]).all()
# El resumen coincide con la tabla de nodos
cuenta = df_nodos.groupby("original_ID").size()
assert df_productos.set_index("original_ID")["total_nodos"].sub(cuenta).eq(0).all()

# Ninguna columna de nutrientes debe llegar al modelo
nutrientes = {"Protein", "Total Fat", "Carbohydrate", "Sugars, total",
              "Fiber, total dietary", "Calcium", "Iron", "Sodium", "Vitamin C",
              "Cholesterol", "Fatty acids, total saturated", "Total Vitamin A"}
assert not (nutrientes & (set(df_nodos.columns) | set(df_productos.columns)))
print("Validaciones OK")

print("\nNodos por producto:")
print(df_productos["total_nodos"].describe())
print("\nProfundidad máxima (cuántos productos hay en cada nivel):")
print(df_productos["profundidad_max"].value_counts().sort_index())
print("\nDistribución del FPro:")
print(df_productos["f_FPro"].describe())
print("\nProductos por tienda:")
print(df_productos["store"].value_counts())
print("\nNodos con nombre vacío:", df_nodos["ingredient_name"].isna().sum(),
      "| ingredientes distintos:", df_nodos["ingredient_name"].nunique())

Validaciones OK

Nodos por producto:
count    22517.000000
mean        18.045477
std         16.574752
min          2.000000
25%          7.000000
50%         14.000000
75%         24.000000
max        194.000000
Name: total_nodos, dtype: float64

Profundidad máxima (cuántos productos hay en cada nivel):
profundidad_max
1    10495
2     9786
3     1863
4      340
5       29
6        3
7        1
Name: count, dtype: int64

Distribución del FPro:
count    22517.000000
mean         0.716286
std          0.235795
min          0.000000
25%          0.593366
50%          0.773611
75%          0.903486
max          0.999056
Name: f_FPro, dtype: float64

Productos por tienda:
store
WholeFoods    10240
Walmart        7175
Target         5102
Name: count, dtype: int64

Nodos con nombre vacío: 0 | ingredientes distintos: 14242


## 7. Guardado

Se guardan las dos tablas para usarlas en el siguiente notebook sin repetir la extracción.

In [8]:
salida = Path("../data/procesado")
salida.mkdir(exist_ok=True)

try:
    df_productos.to_parquet(salida / "productos.parquet", index=False)
    df_nodos.to_parquet(salida / "nodos.parquet", index=False)
    print("Guardado en parquet")
except ImportError:
    df_productos.to_csv(salida / "productos.csv", index=False)
    df_nodos.to_csv(salida / "nodos.csv", index=False)
    print("pyarrow no está instalado: guardado en CSV (pip install pyarrow para parquet)")

pyarrow no está instalado: guardado en CSV (pip install pyarrow para parquet)
